In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import exposure

import zarr
from pylibCZIrw import czi as pyczi

In [2]:
"""Skip during test
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Intermediate")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Proximal")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Distal")
section_folders = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        section_folders.append(folder)
"""

'Skip during test\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Intermediate")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Proximal")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Distal")\nsection_folders = []\nfor folder in series_folder.iterdir():  # Iterate over all items in the folder\n    if folder.is_dir() and folder.name.startswith("S_"):\n        print(f"fould series folder: {folder.name}")\n        section_folders.append(folder)\n'

In [3]:
from atlas.io import is_there_a_single_tif, extract_s_number

In [4]:
series_folder = Path(r"E:\PROJECTS\EM\ATLAS-examples\HHD-S6")
series_folder = Path(r"/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd")
# Find the first CSV file that starts with "PhaseCC_stitching_S_"
csv_file = next(series_folder.glob("phaseCC_stitching_S_*.csv"), None)

if csv_file is None:
    raise FileNotFoundError("No matching CSV file found in the folder.")

print(f"Found CSV file: {csv_file.name}")

# Load the CSV into a DataFrame
df = pd.read_csv(csv_file)

# Try to find a 'pixelSize' column or entry
if 'PixelSizeMicron' in df.columns:
    pix_size_micron =  df['PixelSizeMicron'].max()
else:
    raise ValueError("No 'pixelSize' parameter found in the CSV.")

print(f"pixel_size in microns {pix_size_micron}")


#TODO: important change this to metadata readout
pixel_size = {
    'Value': pix_size_micron,
    'Axial': 0.300,
    'Unit': 'µm'
}
print("work on pixel size")
""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""

tif_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_file() and folder.name.endswith(".tiff"):
        print(f"fould series image: {folder.name}")
        tif_list.append(folder)

Found CSV file: phaseCC_stitching_S_4.csv
pixel_size in microns 0.0200022166698401
work on pixel size
fould series image: stitched_image_S_13.tiff
fould series image: stitched_image_S_15.tiff
fould series image: stitched_image_S_21.tiff
fould series image: stitched_image_S_10.tiff
fould series image: stitched_image_S_18.tiff
fould series image: stitched_image_S_8.tiff
fould series image: stitched_image_S_5.tiff
fould series image: stitched_image_S_11.tiff
fould series image: stitched_image_S_14.tiff
fould series image: stitched_image_S_3.tiff
fould series image: stitched_image_S_9.tiff
fould series image: stitched_image_S_19.tiff
fould series image: stitched_image_S_20.tiff
fould series image: stitched_image_S_4.tiff
fould series image: stitched_image_S_7.tiff
fould series image: stitched_image_S_17.tiff
fould series image: stitched_image_S_6.tiff
fould series image: stitched_image_S_12.tiff
fould series image: stitched_image_S_2.tiff
fould series image: stitched_image_S_16.tiff


In [5]:
"""SKIP during test
tif_list = []
for section_folder in section_folders:
    single_tif, tif_path = is_there_a_single_tif(section_folder)
    if single_tif:
        #print(tif_path)
        tif_list.append(tif_path)
"""

tif_list_sorted = sorted(tif_list, key=extract_s_number)
from atlas.io import reorder_files_by_s_number, parse_shorthand_order

In [6]:
correct_order_path = Path(series_folder.joinpath("correct_order.txt"))

try:
    new_order = parse_shorthand_order(correct_order_path)
    print(f"Loaded correct_order.txt: {new_order}")

    tif_list_sorted = reorder_files_by_s_number(tif_list_sorted, new_order)
    print("Files reordered based on correct_order.txt:")
    for f in tif_list_sorted:
        print(f)

except FileNotFoundError:
    print("No correct_order.txt found — using original order.")
except Exception as e:
    print(f"Error reading or applying correct_order.txt: {e}")

No correct_order.txt found — using original order.


In [7]:
from atlas.io.fibics_metadata import extract_tif_metadata, get_pixel_size_from_tif
from atlas.io import create_empty_folder, rm_tree, apply_alignment, zarr_array_to_czi
from atlas.image_analysis import image_dtype_min_max, rescale_image_intensity
from atlas.alignment import calculate_cumulative_shifts, initialize_alignment_df, pairwise_alignment, first_last_true, ROI, correct_z_alignment_from_points
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation, rescale_image_intensity


In [8]:
output_path = series_folder.joinpath("alignment_results")
create_empty_folder(output_path)

down_scale = 20

# initialize the dataframe, in particular we asign the pair-wise patching of the images
# based on the sorted list of tiff.
z_align_df = initialize_alignment_df(tif_list_sorted, down_scale)
# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_alignment(z_align_df)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# ✅ At the end, save the DataFrame as a CSV for later analysis
z_align_df_path = output_path.joinpath("z_alignment_results.pkl")
z_align_df.to_pickle(z_align_df_path)

z_align_df 

#TODO: implement handling of outliers in the alignment results, this can be done by looking at the current shift and then decide whichones looks weird.

Created new (or emptied) folder: /home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/alignment_results
Processing alignment: ref -> stitched_image_S_2.tiff, moving -> stitched_image_S_2.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [0. 0.]
current shift: [0. 0.]
Processing alignment: ref -> stitched_image_S_2.tiff, moving -> stitched_image_S_3.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [-5.  0.]
current shift: [-5.  0.]
Processing alignment: ref -> stitched_image_S_3.tiff, moving -> stitched_image_S_4.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [ 1. -3.]
current shift: [ 1. -3.]
Processing alignment: ref -> stitched_image_S_4.tiff, moving -> stitched_image_S_5.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [ 2. -3.]
current shift: [ 2. -3.]
Processing alignment: ref -> stitched_image_S_5.tiff, moving -> stitched_image_S_6.tiff
crop pixel shift: [0 0]
Detect

,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 322, 0, 935)","(0, 322, 0, 935)","[0.0, 0.0]","(0, 322, 0, 935)","[0.0, 0.0]",20
1,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 934)","(0, 322, 0, 935)","[-5.0, 0.0]","(0, 321, -5, 929)","[-5.0, 0.0]",20
2,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 322, 0, 935)","(0, 321, 0, 934)","[1.0, -3.0]","(-3, 319, -4, 931)","[-4.0, -3.0]",20
3,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 933)","(0, 322, 0, 935)","[2.0, -3.0]","(-6, 315, -2, 931)","[-2.0, -6.0]",20
4,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 936)","(0, 321, 0, 933)","[-5.0, 0.0]","(-6, 315, -7, 929)","[-7.0, -6.0]",20
5,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 319, 0, 937)","(0, 321, 0, 936)","[9.0, 3.0]","(-3, 316, 2, 939)","[2.0, -3.0]",20
6,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 320, 0, 932)","(0, 319, 0, 937)","[3.0, -4.0]","(-7, 313, 5, 937)","[5.0, -7.0]",20
7,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 935)","(0, 320, 0, 932)","[-5.0, 1.0]","(-6, 315, 0, 935)","[0.0, -6.0]",20
8,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 323, 0, 936)","(0, 321, 0, 935)","[6.0, -4.0]","(-10, 313, 6, 942)","[6.0, -10.0]",20
9,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 323, 0, 934)","(0, 323, 0, 936)","[0.0, 0.0]","(-10, 313, 6, 940)","[6.0, -10.0]",20


In [9]:
z_align_df = pd.read_pickle(z_align_df_path)
z_align_df

,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 322, 0, 935)","(0, 322, 0, 935)","[0.0, 0.0]","(0, 322, 0, 935)","[0.0, 0.0]",20
1,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 934)","(0, 322, 0, 935)","[-5.0, 0.0]","(0, 321, -5, 929)","[-5.0, 0.0]",20
2,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 322, 0, 935)","(0, 321, 0, 934)","[1.0, -3.0]","(-3, 319, -4, 931)","[-4.0, -3.0]",20
3,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 933)","(0, 322, 0, 935)","[2.0, -3.0]","(-6, 315, -2, 931)","[-2.0, -6.0]",20
4,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 936)","(0, 321, 0, 933)","[-5.0, 0.0]","(-6, 315, -7, 929)","[-7.0, -6.0]",20
5,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 319, 0, 937)","(0, 321, 0, 936)","[9.0, 3.0]","(-3, 316, 2, 939)","[2.0, -3.0]",20
6,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 320, 0, 932)","(0, 319, 0, 937)","[3.0, -4.0]","(-7, 313, 5, 937)","[5.0, -7.0]",20
7,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 321, 0, 935)","(0, 320, 0, 932)","[-5.0, 1.0]","(-6, 315, 0, 935)","[0.0, -6.0]",20
8,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 323, 0, 936)","(0, 321, 0, 935)","[6.0, -4.0]","(-10, 313, 6, 942)","[6.0, -10.0]",20
9,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm...,"(0, 323, 0, 934)","(0, 323, 0, 936)","[0.0, 0.0]","(-10, 313, 6, 940)","[6.0, -10.0]",20


In [10]:
zarr.__version__

'3.1.6'

In [11]:
# now we save a downsample version of the alignment for visual inspection
use_down_sample = True
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, percentile_low=2.0, percentile_high=99.95, use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

we will downsample during saving of the zarr, this is good during testing for visual inspection
shape of the zarr array to create: (997, 374, 20)
creating zarr at: /home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/ZEINAB-ROI-w1-20nm-bsd.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18
Done for idx 19
Saving CZI file to: /home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/ZEINAB-ROI-w1-20nm-bsd_ds_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processi

In [12]:
# this is an example of how to apply a manual alignment correction, please ignore if not needed, 
# Define corresponding landmarks after inspecting the initially aligned images.
# Each point uses (z, y, x): the fixed point belongs to the earlier section and
# the matching moving point belongs to the immediately following section. For
# example, [1, 692, 391] and [2, 660, 376] identify the same feature in two
# consecutive sections. Coordinates must use the same pixel scale as the shifts, meaning inspect in the DS data.
# Additional pairs can be appended in matching order; points from the same section
# pair are combined using their median residual correction.
do_correction = False
if do_correction:
    z_align_df = pd.read_pickle(z_align_df_path)
    # index should be given in z, y, x
    list_of_fix_points = [np.array([1, 692., 391.])]
    list_of_mov_points = [np.array([2, 660., 376.])]

    z_align_df = correct_z_alignment_from_points(
        z_align_df,
        list_of_fix_points,
        list_of_mov_points,
    )

    use_down_sample = True
    zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, percentile_low=2.0, percentile_high=99.95, use_down_sample=use_down_sample)

    out_pixel_size = pixel_size.copy()

    if use_down_sample:
        out_pixel_size['Value'] = pixel_size['Value'] * down_scale
        end_str = "_ds_aligned"
    else:
        end_str = "_aligned"

    czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [13]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [14]:
use_down_sample = False
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=1, percentile_low=2.0, percentile_high=99.8, use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

doing full scale alignment, it is recomended to check results before with downscale
shape of the zarr array to create: (19142, 6682, 20)
creating zarr at: /home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/ZEINAB-ROI-w1-20nm-bsd.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18
Done for idx 19
Saving CZI file to: /home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/ZEINAB-ROI-w1-20nm-bsd_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17


In [15]:
if zarr_path.exists():
  rm_tree(zarr_path)